# Dealer Matching on Snowflake — Workshop Notebook

Pipeline steps 1-4: Collect → Locate → Enrich → Resolve

---

## Block 0: Setup
Create WORKSHOP_DB, load DEALERS sample data.

In [ ]:
USE ROLE SYSADMIN;
CREATE DATABASE IF NOT EXISTS WORKSHOP_DB;
USE DATABASE WORKSHOP_DB;
USE SCHEMA PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS WORKSHOP_WH WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE WORKSHOP_WH;
CREATE STAGE IF NOT EXISTS WORKSHOP_STAGE FILE_FORMAT=(TYPE='CSV' FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1);

In [ ]:
CREATE OR REPLACE TABLE DEALERS (
    AGENCY_ID VARCHAR, SITE VARCHAR, AGENCY_NAME VARCHAR,
    ADDRESS VARCHAR, CITY VARCHAR, ZIP_CODE VARCHAR,
    SIRET VARCHAR, MINISITE_URL VARCHAR,
    PRIMARY KEY (AGENCY_ID, SITE)
);

-- Upload first: snow stage copy data/dealers_sample.csv @WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE
COPY INTO DEALERS (AGENCY_ID, SITE, AGENCY_NAME, ADDRESS, CITY, ZIP_CODE, SIRET, MINISITE_URL)
FROM (SELECT $2,$1,$3,$4,$5,$6,$7,$8 FROM @WORKSHOP_STAGE/dealers_sample.csv);

In [ ]:
-- Empty tables for steps 2-3
CREATE OR REPLACE TABLE DEALER_GOOGLE_MAP (
    GOOGLE_PLACE_ID VARCHAR PRIMARY KEY, AI_AGENCY_NAME VARCHAR,
    GOOGLE_JSON VARIANT, CREATED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);
CREATE OR REPLACE TABLE DEALER_ADDRESS_STANDARD (
    AGENCY_ID VARCHAR, SITE VARCHAR, GOOGLE_PLACE_ID VARCHAR, JORECA_ID VARCHAR,
    IS_REUSED BOOLEAN DEFAULT FALSE, AI_JSON VARIANT, AI_STANDARD_ADDRESS VARCHAR,
    CREATED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP(), PRIMARY KEY (AGENCY_ID, SITE)
);

In [ ]:
-- Verify
SELECT 'DEALERS' AS TBL, COUNT(*) AS ROWS FROM DEALERS
UNION ALL SELECT 'DEALER_GOOGLE_MAP', COUNT(*) FROM DEALER_GOOGLE_MAP
UNION ALL SELECT 'DEALER_ADDRESS_STANDARD', COUNT(*) FROM DEALER_ADDRESS_STANDARD;

---
## Block 1: Collect — Explore the data

In [ ]:
SELECT * FROM DEALERS LIMIT 10;

In [ ]:
-- Sites represented
SELECT SITE, COUNT(*) AS CNT FROM DEALERS GROUP BY SITE ORDER BY CNT DESC;

In [ ]:
-- SIRET coverage
SELECT COUNT(*) AS TOTAL,
  SUM(CASE WHEN SIRET!='' AND SIRET IS NOT NULL THEN 1 ELSE 0 END) AS WITH_SIRET,
  ROUND(WITH_SIRET/TOTAL*100,1) AS PCT
FROM DEALERS;

---
## Block 2: Locate — Google Maps via EAI
### Setup (ACCOUNTADMIN)

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE NETWORK RULE google_maps_rule MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('maps.googleapis.com','places.googleapis.com');
CREATE OR REPLACE SECRET google_maps_api_key TYPE=GENERIC_STRING SECRET_STRING='YOUR_GOOGLE_MAPS_API_KEY';
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION google_maps_eai ALLOWED_NETWORK_RULES=(google_maps_rule) ALLOWED_AUTHENTICATION_SECRETS=(google_maps_api_key) ENABLED=TRUE;
GRANT USAGE ON INTEGRATION google_maps_eai TO ROLE SYSADMIN;
USE ROLE SYSADMIN;

In [ ]:
-- Create the Google Maps UDF (see 02-locate-google-maps/google_maps_udf.sql for full code)
-- Paste the CREATE FUNCTION find_google_place(...) from that file here

In [ ]:
-- Test on 3 rows
SELECT d.AGENCY_ID, d.SITE, d.AGENCY_NAME,
  find_google_place(d.AGENCY_NAME, d.ADDRESS, d.CITY, d.ZIP_CODE) AS GMAP
FROM DEALERS d LIMIT 3;

---
## Block 3: Enrich — Gemini via EAI
### Setup (ACCOUNTADMIN)

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE NETWORK RULE gemini_api_rule MODE=EGRESS TYPE=HOST_PORT VALUE_LIST=('generativelanguage.googleapis.com');
CREATE OR REPLACE SECRET gemini_api_key TYPE=GENERIC_STRING SECRET_STRING='YOUR_GEMINI_API_KEY';
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION gemini_eai ALLOWED_NETWORK_RULES=(gemini_api_rule) ALLOWED_AUTHENTICATION_SECRETS=(gemini_api_key) ENABLED=TRUE;
GRANT USAGE ON INTEGRATION gemini_eai TO ROLE SYSADMIN;
USE ROLE SYSADMIN;

In [ ]:
-- Create the Gemini UDF (see 03-enrich-gemini/standardize_address_udf.sql for full code)
-- Paste the CREATE FUNCTION standardize_address(...) from that file here

In [ ]:
-- Test on 3 rows
SELECT d.AGENCY_ID, d.AGENCY_NAME,
  standardize_address(d.AGENCY_NAME, d.ADDRESS, d.CITY, d.ZIP_CODE, NULL, NULL) AS AI
FROM DEALERS d LIMIT 3;

---
## Block 4: Resolve + Results

In [ ]:
-- Enrichment success rate
SELECT COUNT(*) AS TOTAL,
  SUM(CASE WHEN AI_STANDARD_ADDRESS IS NOT NULL THEN 1 ELSE 0 END) AS ENRICHED,
  ROUND(ENRICHED/TOTAL*100,1) AS PCT
FROM DEALER_ADDRESS_STANDARD;

In [ ]:
-- Before / after
SELECT d.AGENCY_NAME AS RAW_NAME,
  das.AI_JSON:"clean_agency_name"::VARCHAR AS CLEAN_NAME,
  d.ADDRESS AS RAW_ADDR,
  das.AI_STANDARD_ADDRESS AS STD_ADDR,
  das.AI_JSON:"confidence"::FLOAT AS CONF
FROM DEALERS d
JOIN DEALER_ADDRESS_STANDARD das ON d.AGENCY_ID=das.AGENCY_ID AND d.SITE=das.SITE
ORDER BY CONF DESC LIMIT 20;

---
## Done!

Pipeline steps 1-4 running in Snowflake. No Linux server, no batch files.

**Next steps (optional modules in the repo):**
- `optional-05-matching/` — Multi-level matching → CONCESSION
- `optional-06-cortex-ai/` — Compare with Snowflake Cortex AI
- `optional-07-data-quality/` — DMFs for automated quality checks
- `optional-08-change-tracking/` — AGENCY_CHANGE_LOG
- `optional-09-scheduling/` — Snowflake Tasks